# Map Data Cleaning

This notebook prepares map-ready dyslexia support data for Australian parents of primary school children. It reads the two raw source datasets, filters out closed and irrelevant services, classifies each record into the agreed categories, geocodes missing coordinates when address information exists, and exports the final `map_data.csv` file.

Core rules used throughout this notebook:
- Closed institutions are excluded.
- Missing values are standardized to `To be added`.
- Missing coordinates are geocoded from address information where possible.

## 1. Load Libraries and File Paths

This block imports the libraries used in cleaning, classification, and geocoding. It also defines the source files, output file, and shared constants so the rest of the notebook uses a single configuration.

Expected output: the notebook is ready to read the raw CSVs and export a standardized dataset.

In [1]:
# Step 1 code note: import tools and define the source/output file paths used by the whole notebook.
from __future__ import annotations

# json is used to read the response returned by the geocoding website.
import json

# re is used for text cleaning and keyword patterns.
import re

# time is used to pause briefly between geocoding requests.
import time

# urllib is used to send simple web requests for geocoding.
import urllib.parse
import urllib.request

# pathlib makes file paths easier to read and safer than plain strings.
from pathlib import Path

# pandas is used to read, clean, filter, combine, and export CSV data.
import pandas as pd

# display shows dataframes nicely inside the notebook output.
from IPython.display import display

# This text is used whenever a field is missing.
# Example: if phone number is missing, the final CSV will show "To be added".
PLACEHOLDER = 'To be added'

# Nominatim asks requests to include a user agent.
# This is only a technical request header, not user tracking or login.
USER_AGENT = 'dyslexia-map-data-prep/1.0'

# If the notebook is opened from data/map, Path.cwd().name will be "map".
# If the notebook is run from the project root, we manually point to data/map.
DATA_DIR = Path.cwd()
if DATA_DIR.name != 'map':
    DATA_DIR = Path.cwd() / 'data' / 'map'

# These are the two original source files.
# We only read them; we do not edit them.
HEALTHDIRECT_PATH = DATA_DIR / 'healthdirect_nhsd_services_directory_2025.csv'
DATADOTGOV_PATH = DATA_DIR / 'datadotgov_main.csv'

# This is the final cleaned file that the notebook exports.
OUTPUT_PATH = DATA_DIR / 'map_data.csv'

# These columns define the final structure of map_data.csv.
# Keeping this list here makes sure every output row has the same fields.
FINAL_COLUMNS = [
    'name', 'category', 'service_kind', 'profession_type', 'address', 'suburb', 'state',
    'postcode', 'latitude', 'longitude', 'phone', 'website', 'source', 'description', 'highlighted'
]

# These settings make dataframe previews wider and easier to inspect.
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)

print('Healthdirect path:', HEALTHDIRECT_PATH)
print('Data.gov path:', DATADOTGOV_PATH)
print('Output path:', OUTPUT_PATH)

Healthdirect path: /Users/datong/Documents/5120/Nurodiversity inclusive design/dysLe_TP10/data/map/healthdirect_nhsd_services_directory_2025.csv
Data.gov path: /Users/datong/Documents/5120/Nurodiversity inclusive design/dysLe_TP10/data/map/datadotgov_main.csv
Output path: /Users/datong/Documents/5120/Nurodiversity inclusive design/dysLe_TP10/data/map/map_data.csv


## 2. Load Source Datasets

This block reads the two raw files and gives a quick overview of their shape, column names, and example rows. The goal is to help teammates understand what each source contains before any filtering happens.

Expected output: row counts, column lists, and small previews for both datasets.

In [2]:
# Step 2 code note: load both raw CSV files so we can inspect them before cleaning.
# Read the Healthdirect file into a dataframe.
# This dataset is used mainly for medical, psychology, speech, and occupational therapy services.
healthdirect_raw = pd.read_csv(HEALTHDIRECT_PATH)

# Read the ACNC/data.gov file into a dataframe.
# This dataset is used mainly for dyslexia associations and community support organisations.
datadotgov_raw = pd.read_csv(DATADOTGOV_PATH)

# Show how many rows and columns are in each raw dataset.
# This helps us understand the size of each source before filtering.
print('Healthdirect shape:', healthdirect_raw.shape)
print('Data.gov shape:', datadotgov_raw.shape)

# Show all Healthdirect column names so teammates can see what fields are available.
print('\nHealthdirect columns:')
display(pd.Series(healthdirect_raw.columns))

# Show all Data.gov column names so teammates can see what fields are available.
print('\nData.gov columns:')
display(pd.Series(datadotgov_raw.columns))

# Preview a few Healthdirect rows before cleaning.
print('\nHealthdirect preview:')
display(healthdirect_raw.head(3))

# Preview a few Data.gov rows before cleaning.
print('\nData.gov preview:')
display(datadotgov_raw.head(3))

/var/folders/xy/3853xvsd0xb3mwrdg3w9cwnr0000gn/T/ipykernel_87029/2127981801.py:4: DtypeWarning: Columns (20,21,22,23,24,25,26,27,28,29,30,31,32,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  healthdirect_raw = pd.read_csv(HEALTHDIRECT_PATH)


Healthdirect shape: (129231, 66)
Data.gov shape: (65294, 69)

Healthdirect columns:


/var/folders/xy/3853xvsd0xb3mwrdg3w9cwnr0000gn/T/ipykernel_87029/2127981801.py:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  datadotgov_raw = pd.read_csv(DATADOTGOV_PATH)


0                                   FID
1                       nhsd_service_id
2                                status
3                          organization
4                       organization_id
                    ...                
61    availability_royal_hobart_regatta
62           availability_devonport_cup
63          availability_launceston_cup
64        availability_king_island_show
65                             the_geom
Length: 66, dtype: object


Data.gov columns:


0                          ABN
1           Charity_Legal_Name
2     Other_Organisation_Names
3                 Address_Type
4               Address_Line_1
                ...           
64        Victims_of_Disasters
65                       Youth
66                     animals
67                 environment
68     other_gender_identities
Length: 69, dtype: object


Healthdirect preview:


,FID,nhsd_service_id,status,organization,organization_id,monday_open_hours,tuesday_open_hours,wednesday_open_hours,thursday_open_hours,friday_open_hours,address,city,state,country,postcode,latitude,longitude,saturday_open_hours,parent_organization_id,parent_organization_name,availability_anzac_day,availability_australia_day,availability_easter_sunday,availability_easter_saturday,availability_easter_monday,...,availability_canberra_day,availability_kings_birthday_act,availability_labour_day_act,availability_reconciliation_day,availability_adelaide_cup_day,availability_kings_birthday_sa,availability_labour_day_sa,availability_eight_hours_day,availability_agfest,availability_kings_birthday_tas,availability_burnie_show,availability_launceston_show,availability_flinders_island_show,availability_royal_hobart_show,availability_recreation_day,availability_devonport_show,availability_may_day,availability_picnic_day,availability_alternate_hours,availability_kings_birthday_nt,availability_royal_hobart_regatta,availability_devonport_cup,availability_launceston_cup,availability_king_island_show,the_geom
0,healthdirect_nhsd_services_directory_2025.164,e3f1ecbd-a5bf-cf69-6b7b-98d7616af124,UNKNOWN,Linda Clifton - Psychology,3db89e9d-6b66-9af2-2528-ed497245a135,NaN,NaN,NaN,NaN,NaN,Confidential Address,MALVERN,VIC,AUS,3144.0,-37.857281,145.034500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-37.85728073120117 145.0345001220703)
1,healthdirect_nhsd_services_directory_2025.277,8cdf43b2-7db3-78ce-1532-8377adecff1d,UNKNOWN,General Dental,7b4956e4-93e8-6b51-3730-d7e7c02f586d,NaN,NaN,NaN,NaN,NaN,10 William Street,EARLWOOD,NSW,AUS,2206.0,-33.927578,151.123627,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-33.92757797241211 151.12362670898438)
2,healthdirect_nhsd_services_directory_2025.308,771c9ab2-1f1d-4380-890f-005ac5c578f1,UNKNOWN,Dr Paul Cozzi - Miranda,4d439e1f-1214-49d8-b067-1b3a56ff5488,NaN,NaN,NaN,NaN,NaN,"Level 3,531 Kingsway Way",MIRANDA,NSW,AUS,2228.0,-34.034044,151.104088,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-34.034044 151.104088)



Data.gov preview:


,ABN,Charity_Legal_Name,Other_Organisation_Names,Address_Type,Address_Line_1,Address_Line_2,Address_Line_3,Town_City,State,Postcode,Country,Charity_Website,Registration_Date,Date_Organisation_Established,Charity_Size,Number_of_Responsible_Persons,Financial_Year_End,Operates_in_ACT,Operates_in_NSW,Operates_in_NT,Operates_in_QLD,Operates_in_SA,Operates_in_TAS,Operates_in_VIC,Operates_in_WA,...,Communities_Overseas,Early_Childhood,Ethnic_Groups,Families,Females,Financially_Disadvantaged,LGBTIQA+,General_Community_in_Australia,Males,Migrants_Refugees_or_Asylum_Seekers,Other_Beneficiaries,Other_Charities,People_at_risk_of_homelessness,People_with_Chronic_Illness,People_with_Disabilities,Pre_Post_Release_Offenders,Rural_Regional_Remote_Communities,Unemployed_Person,Veterans_or_their_families,Victims_of_crime,Victims_of_Disasters,Youth,animals,environment,other_gender_identities
0,9.783047e+10,The Dale Family Foundation,NaN,Business,L 15 216 St Georges Tce,NaN,NaN,Perth,WA,6000,Australia,NaN,13/11/2024,13/11/2024,NaN,2,30-Jun,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9.017506e+10,The SJD Homes Foundation,NaN,Business,433 Princes Hwy,NaN,NaN,Officer,VIC,3809,Australia,NaN,27/10/2024,27/10/2024,Small,7,30-Jun,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,...,NaN,NaN,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.531376e+10,NORTH WEST REGIONAL LANDCARERS INC,NaN,Business,Po Box 3122,NaN,NaN,West Tamworth,NSW,2340,Australia,www.nwrl.org.au,11/10/2023,11/10/2023,Medium,5,30-Jun,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Y,NaN,NaN,Y,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN,NaN,NaN,Y,Y,Y,Y


## 3. Inspect Relevant Columns

This block narrows the view to the columns actually used in the cleaning workflow. It lets the team verify that the required status, location, and naming fields are present before classification and geocoding.

Expected output: concise previews of the fields used for filtering, classification, and export.

In [3]:
# Step 3 code note: show only the columns that matter for filtering, classification, and mapping.
# These are the Healthdirect fields we use later.
# status tells us whether a service is closed.
# organization gives the service name.
# address/city/state/postcode/latitude/longitude are needed for map pins.
healthdirect_cols = [
    'status', 'organization', 'address', 'city', 'state', 'postcode', 'latitude', 'longitude'
]

# These are the Data.gov fields we use later.
# Charity_Legal_Name and Other_Organisation_Names help us find dyslexia-related organisations.
# Address fields help us geocode community organisations.
# Charity_Website gives a contact method when phone is missing.
datadotgov_cols = [
    'Charity_Legal_Name', 'Other_Organisation_Names', 'Address_Line_1', 'Town_City',
    'State', 'Postcode', 'Charity_Website'
]

# Show only useful Healthdirect columns.
display(healthdirect_raw[healthdirect_cols].head(10))

# Show only useful Data.gov columns.
display(datadotgov_raw[datadotgov_cols].head(10))

,status,organization,address,city,state,postcode,latitude,longitude
0,UNKNOWN,Linda Clifton - Psychology,Confidential Address,MALVERN,VIC,3144.0,-37.857281,145.034500
1,UNKNOWN,General Dental,10 William Street,EARLWOOD,NSW,2206.0,-33.927578,151.123627
2,UNKNOWN,Dr Paul Cozzi - Miranda,"Level 3,531 Kingsway Way",MIRANDA,NSW,2228.0,-34.034044,151.104088
3,UNKNOWN,Mount Martha Osteopathy,6 Reeve Street,MOUNT MARTHA,VIC,3934.0,-38.268791,145.018662
4,CLOSED,Dr John H Drew,59A Doncaster Road,BALWYN NORTH,VIC,3104.0,-37.792248,145.070480
5,CLOSED,Runcorn Plaza Family Medical Practice,"Shop 1,258 Warrigal Road",RUNCORN,QLD,4113.0,-27.588823,153.084290
6,CLOSED,Mansfield Dental Group,132 High Street,MANSFIELD,VIC,3722.0,-37.051888,146.083817
7,CLOSED,Dandenong Podiatry,71 Cleeland Street,DANDENONG,VIC,3175.0,-37.980564,145.214996
8,CLOSED,Brunswick Health Clinic,68 Melville Road,BRUNSWICK WEST,VIC,3055.0,-37.763180,144.944321
9,CLOSED,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Charity_Legal_Name,Other_Organisation_Names,Address_Line_1,Town_City,State,Postcode,Charity_Website
0,The Dale Family Foundation,NaN,L 15 216 St Georges Tce,Perth,WA,6000,NaN
1,The SJD Homes Foundation,NaN,433 Princes Hwy,Officer,VIC,3809,NaN
2,NORTH WEST REGIONAL LANDCARERS INC,NaN,Po Box 3122,West Tamworth,NSW,2340,www.nwrl.org.au
3,Catholic Cemeteries & Crematoria Trust,NaN,L 2 11 Murray Rose Ave,Sydney Olympic Park,NSW,2127,https://catholiccemeteries.com.au/
4,HOPE ARABIC CHURCH INCORPORATED,"Hope Arabic Church - South Australia, كنيسة ال...",614 Burbridge Rd,West Beach,SA,5024,www.hopearabicchurch.com
5,Frederick Miller Community Garden,NaN,Frederick Miller Reserve,Semaphore Park,SA,5019,NaN
6,DELATITE LANDCARE GROUP INC,NaN,24 Highett St,Mansfield,VIC,3722,www.up2us.org.au
7,TELCO TOGETHER FOUNDATION,NaN,L 10 452 Flinders St,Melbourne,VIC,3000,telcotogether.org
8,TECHNION AUSTRALIA PBI LIMITED,NaN,L 9 151 Macquarie St,Sydney,NSW,2000,NaN
9,CAIRNS MAR THOMA CHURCH INCORPORATED,NaN,19 Tierney St,Innisfail Estate,QLD,4860,NaN


## 4. Define Cleaning Helpers

This block contains all helper functions used by the notebook. They clean text, normalize websites, classify organizations, build addresses, geocode missing coordinates, and deduplicate the final records. Keeping the logic here makes the later steps easier to read.

Expected output: reusable functions for the remaining workflow.

In [4]:
# Step 4 code note: define small helper functions so repeated cleaning logic is easier to reuse.
def clean(value):
    # Convert missing values like NaN into an empty string.
    # This avoids errors when we call string methods later.
    if pd.isna(value):
        return ''

    # Convert the value to text, remove extra spaces, tabs, and line breaks.
    return re.sub(r'\s+', ' ', str(value)).strip()


def clean_postcode(value):
    # Postcodes should be text, not decimal numbers.
    # Sometimes pandas reads postcode values as 3122.0 instead of 3122.
    postcode = clean(value)

    # Remove a trailing .0 only when the whole value is a number like 3122.0.
    if re.fullmatch(r'\d+\.0', postcode):
        postcode = postcode[:-2]

    return postcode


def title_case_words(value):
    # Some suburb names are all uppercase in the source file, such as MELBOURNE.
    # This function makes them easier to read, such as Melbourne.
    text = clean(value)
    return ' '.join(part.capitalize() if part.isupper() else part for part in text.split())


def normalize_website(url):
    # Clean the website field first.
    url = clean(url)

    # If the source does not provide a website, use the required fallback text.
    if not url:
        return PLACEHOLDER

    # If the website does not start with http:// or https://, add https://.
    # This makes the website easier to open later from the frontend.
    if not re.match(r'^https?://', url, flags=re.IGNORECASE):
        return f'https://{url}'
    return url


def build_address(*parts):
    # Combine address parts into one readable address string.
    # Empty parts are skipped so we do not get repeated commas.
    values = [clean(part) for part in parts if clean(part)]

    # If every address part is empty, keep the fallback text.
    return ', '.join(values) if values else PLACEHOLDER


def classify_healthdirect(name):
    # This function decides whether a Healthdirect row is useful for our map.
    # It returns category, service kind, and profession type.
    lowered = clean(name).lower()
    if not lowered:
        return None

    # These words usually describe services outside our target audience or project focus.
    # If one appears in the service name, we exclude the row.
    blacklist = [
        'antenatal', 'postnatal', 'aged care', 'cardiac rehab', 'driver assessment',
        'rehabilitation consultants', 'rehabilitation service', 'bulk billed',
        'hospital - occupational therapy', 'primary health centre - psychology'
    ]
    if any(term in lowered for term in blacklist):
        return None

    # Assessment category: services that may help identify or assess learning difficulties.
    if 'educational psychologist' in lowered or 'educational psychology' in lowered:
        return ('Assessment', 'Medical/Psych', 'Educational Psychologist')
    if 'paediatrician' in lowered or 'pediatrician' in lowered:
        return ('Assessment', 'Medical/Psych', 'Paediatrician')
    if 'psychologist' in lowered or 'psychology' in lowered:
        return ('Assessment', 'Medical/Psych', 'Psychologist')

    # Intervention category: services that may help with communication, reading, writing, or fine-motor support.
    if 'speech pathology' in lowered or 'speech patholog' in lowered or 'speech and language' in lowered:
        return ('Intervention', 'Education', 'Speech Pathologist')
    if 'occupational therapist' in lowered or 'occupational therapy' in lowered:
        return ('Intervention', 'Education', 'Occupational Therapist')

    # If no useful keyword is found, exclude the row.
    return None


def classify_datadotgov(name):
    # This function classifies charity/community organisations.
    lowered = clean(name).lower()

    # SPELD, DSF, and AUSPELD are trusted dyslexia-related organisations.
    # We mark them as highlighted so the map can show them differently.
    highlighted = any(term in lowered for term in ['speld', 'dsf', 'auspeld'])

    # Choose a readable profession/type label for the popup.
    if 'learning difficulties' in lowered:
        profession = 'Learning Difficulties Association'
    elif any(term in lowered for term in ['dyslexia', 'speld', 'dsf', 'auspeld']):
        profession = 'Dyslexia Support Organisation'
    else:
        profession = 'Community Support Organisation'

    return ('Support Groups', 'Community', profession, 'Yes' if highlighted else 'No')


def geocode_queries(address, suburb, state, postcode):
    # This function builds possible search strings for geocoding.
    # We try a detailed address first, then broader locations if needed.
    address = clean(address)
    suburb = clean(suburb)
    state = clean(state)
    postcode = clean(postcode)

    queries = []

    # PO Boxes are mailing addresses, not physical places, so they are weak for map pins.
    # For PO Boxes, we skip the full address and try suburb/state/postcode instead.
    if address and not address.lower().startswith(('po box', 'p.o. box')):
        queries.append(build_address(address, suburb, state, postcode, 'Australia'))

    # Broader fallback query using suburb/state/postcode.
    if suburb or state or postcode:
        queries.append(build_address(suburb, state, postcode, 'Australia'))

    # Last fallback if suburb is missing but state or postcode exists.
    if state or postcode:
        queries.append(build_address(state, postcode, 'Australia'))

    # Remove duplicate queries while keeping the original order.
    return list(dict.fromkeys([query for query in queries if query != PLACEHOLDER]))


def geocode(query, cache):
    # If this exact query was already tried in this notebook run, reuse the result.
    # This avoids sending the same request multiple times.
    if query in cache:
        return cache[query]

    # Build the geocoding URL. Nominatim returns latitude and longitude for an address string.
    url = 'https://nominatim.openstreetmap.org/search?' + urllib.parse.urlencode({
        'q': query, 'format': 'jsonv2', 'limit': 1
    })

    # The request needs headers so the geocoding service accepts it.
    request = urllib.request.Request(
        url,
        headers={'User-Agent': USER_AGENT, 'Accept-Language': 'en'},
    )

    # If the request fails, keep the row unresolved instead of stopping the whole notebook.
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            payload = json.loads(response.read().decode('utf-8'))
    except Exception:
        payload = []

    # Use the first geocoding result if one exists.
    if payload:
        result = (payload[0]['lat'], payload[0]['lon'])
    else:
        result = (PLACEHOLDER, PLACEHOLDER)

    # Store the result in memory for this run.
    cache[query] = result

    # Pause briefly so we do not send requests too quickly.
    time.sleep(1)
    return result


def completeness_score(row):
    # This score helps choose the better row if duplicates exist.
    # A row with coordinates, website, and description is more complete.
    fields = ['latitude', 'longitude', 'phone', 'website', 'description']
    return sum(1 for field in fields if row[field] != PLACEHOLDER)


def dedupe_records(df):
    # This removes duplicate records based on name + address + state.
    # If duplicates exist, keep the one with more complete information.
    records = df.to_dict(orient='records')
    best = {}
    for row in records:
        key = '|'.join([clean(row['name']).lower(), clean(row['address']).lower(), clean(row['state']).lower()])
        if key not in best or completeness_score(row) > completeness_score(best[key]):
            best[key] = row

    deduped = pd.DataFrame(best.values())
    return deduped.sort_values(['category', 'name']).reset_index(drop=True)


def apply_placeholder(df, columns):
    # This makes missing values consistent across the final CSV.
    # Instead of blanks, the output will show "To be added".
    for column in columns:
        df[column] = df[column].fillna('').apply(clean)
        df[column] = df[column].replace('', PLACEHOLDER)
    return df

## 5. Clean and Filter Healthdirect

This block removes closed services and keeps only rows whose organization names match the targeted assessment and intervention keywords. It also excludes obviously irrelevant services such as aged-care and similar non-child support entries.

### First filter: remove closed services
The notebook checks the `status` column.

Rows are removed if:
- `status = CLOSED`

Rows can continue to the next step if:
- `status = OPEN`
- `status = UNKNOWN`
- status is not marked as `CLOSED`

This is because the map should not recommend services that are already closed.

### Second filter: keep relevant Healthdirect keywords
The notebook searches the `organization` name for these keywords:

| Keyword used | Why it is included |
|---|---|
| `psychologist` | May provide child psychology or learning-difficulty assessment. |
| `psychology` | May be a psychology clinic or psychology service. |
| `educational` | May indicate educational psychology, which is relevant to learning assessment. |
| `paediatrician` | Paediatricians may support child development or learning-difficulty assessment. |
| `speech` | May indicate speech/language support, which can relate to reading and language development. |
| `speech pathology` | Speech pathology is a relevant intervention support area. |
| `occupational therapy` | Occupational therapy may support handwriting, fine-motor skills, and learning participation. |
| `occupational therapist` | Occupational therapists may provide practical child support. |

Only rows that match these keywords continue to the classification step.

### Third filter: remove clearly irrelevant services
Some rows match the broad keywords but are not suitable for this dyslexia-support map. The notebook removes rows containing these terms:

| Excluded term | Why it is excluded |
|---|---|
| `antenatal` | Pregnancy-related service, not child dyslexia support. |
| `postnatal` | Post-birth parent/baby service, not child dyslexia support. |
| `aged care` | Older-adult care, outside the target audience. |
| `cardiac rehab` | Heart rehabilitation, unrelated to dyslexia support. |
| `driver assessment` | Driving assessment, unrelated to children aged 6-8. |
| `rehabilitation consultants` | Too broad and usually not dyslexia-specific. |
| `rehabilitation service` | Too broad and usually not dyslexia-specific. |
| `bulk billed` | Payment information, not evidence that the service is dyslexia-related. |
| `hospital - occupational therapy` | Usually a broad hospital OT service, not necessarily child/dyslexia specific. |
| `primary health centre - psychology` | Too broad and not clearly dyslexia-related. |

### Fourth filter: require usable location information
Rows are also removed if they have no usable location fields at all.

The notebook checks these fields:
- `address`
- `city`
- `state`
- `postcode`

Expected output: a smaller Healthdirect candidate set focused on usable dyslexia-related support points.

In [5]:
# Step 5 code note: filter Healthdirect to active, relevant assessment/intervention candidates only.
healthdirect = healthdirect_raw.copy()
healthdirect['status_clean'] = healthdirect['status'].apply(clean).str.upper()
healthdirect['organization_clean'] = healthdirect['organization'].apply(clean)

healthdirect = healthdirect[healthdirect['status_clean'] != 'CLOSED'].copy()

keywords = [
    'psychologist', 'psychology', 'educational', 'paediatrician',
    'speech', 'speech pathology', 'occupational therapy', 'occupational therapist'
]
pattern = '|'.join(re.escape(keyword) for keyword in keywords)
healthdirect = healthdirect[
    healthdirect['organization_clean'].str.lower().str.contains(pattern, na=False)
].copy()

healthdirect['classification'] = healthdirect['organization_clean'].apply(classify_healthdirect)
healthdirect = healthdirect[healthdirect['classification'].notna()].copy()

location_mask = healthdirect[['address', 'city', 'state', 'postcode']].fillna('').map(clean).ne('').any(axis=1)
healthdirect = healthdirect[location_mask].copy()

print('Filtered Healthdirect rows:', len(healthdirect))
display(healthdirect[['organization', 'status', 'address', 'city', 'state', 'postcode']].head(20))

Filtered Healthdirect rows: 172


,organization,status,address,city,state,postcode
0,Linda Clifton - Psychology,UNKNOWN,Confidential Address,MALVERN,VIC,3144.0
27,Brisbane Waters Speech Pathology,UNKNOWN,"Suite 7a,188 The Entrance Road",ERINA,NSW,2250.0
58,"Kemp, Jane - Psychologist",UNKNOWN,104 Rose Terrace,WAYVILLE,SA,5034.0
73,Beecroft Speech Pathology,UNKNOWN,21 Grace Avenue,BEECROFT,NSW,2119.0
866,Occupational Therapy Mackay,UNKNOWN,37 BRISBANE STREET,MACKAY,QLD,4740.0
1510,Mr Harry Constantinou - Psychologist,UNKNOWN,302 Albert Road,SOUTH MELBOURNE,VIC,3205.0
3015,Dr Robert Woodfield - Psychologist,UNKNOWN,"Norwest Private Hospital,Suite 110,9 Norbrik D...",BELLA VISTA,NSW,2153.0
3159,Phillips Clinical Psychology,UNKNOWN,"Suite 14,296 Marrickville Road",MARRICKVILLE,NSW,2204.0
3348,Inner West Speech Pathology,UNKNOWN,71 Clarence Street,BELFIELD,NSW,2191.0
3768,Sally-Ann E. Gordon Speech Pathologist,UNKNOWN,18 Rawson Penfold Drive,ROSSLYN PARK,SA,5072.0


## 6. Classify Healthdirect Records

This block converts the filtered Healthdirect rows into the shared map schema. The category and service kind come from fixed rules so teammates can see exactly how the classification is applied.

### `category`
`category` describes the **parent's need**. This is the field used for the main map filters.

Allowed values in the whole project:
- `Assessment`: services that may help identify, assess, or diagnose learning difficulties.
- `Intervention`: services that may help support reading, writing, language, or fine-motor needs.
- `Support Groups`: organisations that provide community information, advocacy, parent support, or dyslexia-specific guidance.

For Healthdirect rows, this notebook only creates:
- `Assessment`
- `Intervention`

Healthdirect does not provide the main community/support-group records in this workflow.

### `service_kind`
`service_kind` describes the **type of provider shown by the map pin colour**. It is different from `category`.

For Healthdirect rows:
- `Medical/Psych` is used for psychology, educational psychology, and paediatrician records.
- `Education` is used for speech pathology and occupational therapy records.

This lets the map do two things at once:
- filter by parent need using `category`
- colour pins by provider type using `service_kind`

### Healthdirect classification mapping

| Matched text in `organization` | `category` | `service_kind` | `profession_type` |
|---|---|---|---|
| `educational psychologist` | `Assessment` | `Medical/Psych` | `Educational Psychologist` |
| `educational psychology` | `Assessment` | `Medical/Psych` | `Educational Psychologist` |
| `paediatrician` | `Assessment` | `Medical/Psych` | `Paediatrician` |
| `pediatrician` | `Assessment` | `Medical/Psych` | `Paediatrician` |
| `psychologist` | `Assessment` | `Medical/Psych` | `Psychologist` |
| `psychology` | `Assessment` | `Medical/Psych` | `Psychologist` |
| `speech pathology` | `Intervention` | `Education` | `Speech Pathologist` |
| `speech patholog` | `Intervention` | `Education` | `Speech Pathologist` |
| `speech and language` | `Intervention` | `Education` | `Speech Pathologist` |
| `occupational therapist` | `Intervention` | `Education` | `Occupational Therapist` |
| `occupational therapy` | `Intervention` | `Education` | `Occupational Therapist` |


Expected output: a standardized Healthdirect dataframe ready to merge later.

In [6]:
# Step 6 code note: reshape Healthdirect rows into the final shared map_data schema.
healthdirect[['category', 'service_kind', 'profession_type']] = pd.DataFrame(
    healthdirect['classification'].tolist(), index=healthdirect.index
)

healthdirect_prepared = pd.DataFrame({
    'name': healthdirect['organization_clean'],
    'category': healthdirect['category'],
    'service_kind': healthdirect['service_kind'],
    'profession_type': healthdirect['profession_type'],
    'address': healthdirect.apply(
        lambda row: build_address(row.get('address'), title_case_words(row.get('city')), clean(row.get('state')).upper(), clean_postcode(row.get('postcode'))),
        axis=1,
    ),
    'suburb': healthdirect['city'].apply(title_case_words),
    'state': healthdirect['state'].apply(lambda value: clean(value).upper()),
    'postcode': healthdirect['postcode'].apply(clean_postcode),
    'latitude': healthdirect['latitude'].apply(clean),
    'longitude': healthdirect['longitude'].apply(clean),
    'phone': PLACEHOLDER,
    'website': PLACEHOLDER,
    'source': 'Healthdirect NHSD Services Directory 2025',
    'description': healthdirect['category'].map({
        'Assessment': 'Assessment support identified from Healthdirect NHSD Services Directory 2025.',
        'Intervention': 'Intervention support identified from Healthdirect NHSD Services Directory 2025.',
    }),
    'highlighted': 'No',
})

healthdirect_prepared = apply_placeholder(healthdirect_prepared, ['name', 'profession_type', 'address', 'suburb', 'state', 'postcode', 'latitude', 'longitude', 'phone', 'website', 'description'])
display(healthdirect_prepared.head(10))
print(healthdirect_prepared['category'].value_counts())

,name,category,service_kind,profession_type,address,suburb,state,postcode,latitude,longitude,phone,website,source,description,highlighted
0,Linda Clifton - Psychology,Assessment,Medical/Psych,Psychologist,"Confidential Address, Malvern, VIC, 3144",Malvern,VIC,3144,-37.85728073,145.03450012,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
27,Brisbane Waters Speech Pathology,Intervention,Education,Speech Pathologist,"Suite 7a,188 The Entrance Road, Erina, NSW, 2250",Erina,NSW,2250,-33.43602371,151.38575745,To be added,To be added,Healthdirect NHSD Services Directory 2025,Intervention support identified from Healthdir...,No
58,"Kemp, Jane - Psychologist",Assessment,Medical/Psych,Psychologist,"104 Rose Terrace, Wayville, SA, 5034",Wayville,SA,5034,-34.94286346,138.58792114,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
73,Beecroft Speech Pathology,Intervention,Education,Speech Pathologist,"21 Grace Avenue, Beecroft, NSW, 2119",Beecroft,NSW,2119,-33.74789429,151.05126953,To be added,To be added,Healthdirect NHSD Services Directory 2025,Intervention support identified from Healthdir...,No
866,Occupational Therapy Mackay,Intervention,Education,Occupational Therapist,"37 BRISBANE STREET, Mackay, QLD, 4740",Mackay,QLD,4740,-21.1447693,149.18829181,To be added,To be added,Healthdirect NHSD Services Directory 2025,Intervention support identified from Healthdir...,No
1510,Mr Harry Constantinou - Psychologist,Assessment,Medical/Psych,Psychologist,"302 Albert Road, South Melbourne, VIC, 3205",South Melbourne,VIC,3205,-37.83887482,144.9629364,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
3015,Dr Robert Woodfield - Psychologist,Assessment,Medical/Psych,Psychologist,"Norwest Private Hospital,Suite 110,9 Norbrik D...",Bella Vista,NSW,2153,-33.74521637,150.95401001,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
3159,Phillips Clinical Psychology,Assessment,Medical/Psych,Psychologist,"Suite 14,296 Marrickville Road, Marrickville, ...",Marrickville,NSW,2204,-33.91053391,151.15592957,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
3348,Inner West Speech Pathology,Intervention,Education,Speech Pathologist,"71 Clarence Street, Belfield, NSW, 2191",Belfield,NSW,2191,-33.9053688,151.08631897,To be added,To be added,Healthdirect NHSD Services Directory 2025,Intervention support identified from Healthdir...,No
3768,Sally-Ann E. Gordon Speech Pathologist,Intervention,Education,Speech Pathologist,"18 Rawson Penfold Drive, Rosslyn Park, SA, 5072",Rosslyn Park,SA,5072,-34.91970825,138.68041992,To be added,To be added,Healthdirect NHSD Services Directory 2025,Intervention support identified from Healthdir...,No


category
Assessment      86
Intervention    86
Name: count, dtype: int64


## 7. Clean and Filter Data.gov / ACNC Records

This block identifies highly relevant community and support organizations from the ACNC extract. It uses focused name matching so the support-group layer stays small, trustworthy, and dyslexia-specific.

### Fields used for matching
The notebook searches across these fields:
- `Charity_Legal_Name`
- `Other_Organisation_Names`

These are used because the organisation name is the clearest signal of whether the charity is related to dyslexia or learning difficulties.

### Data.gov / ACNC keywords kept

| Keyword used | Why it is included |
|---|---|
| `dyslexia` | Directly related to the project topic. |
| `speld` | SPELD organisations are important dyslexia / specific learning difficulty support organisations in Australia. |
| `learning difficulties` | Directly related to children with learning challenges. |
| `specific learning` | Helps capture organisations related to specific learning difficulties. |

Rows that do not match these keywords are excluded from the support-group layer.

### Why this dataset is used differently from Healthdirect
Healthdirect is used for individual services such as psychology, speech pathology, and occupational therapy.

Data.gov / ACNC is used for organisations such as:
- dyslexia associations
- learning-difficulty organisations
- community support groups
- advocacy organisations
- parent information/support organisations

Expected output: a curated support-group candidate set from the charity dataset.

In [7]:
# Step 7 code note: filter the ACNC/data.gov charity file to dyslexia-related support organisations.
datadotgov = datadotgov_raw.copy()
datadotgov['legal_name_clean'] = datadotgov['Charity_Legal_Name'].apply(clean)
datadotgov['other_names_clean'] = datadotgov['Other_Organisation_Names'].apply(clean)

keywords = ['dyslexia', 'speld', 'learning difficulties', 'specific learning']
pattern = '|'.join(re.escape(keyword) for keyword in keywords)
mask = (datadotgov['legal_name_clean'] + ' | ' + datadotgov['other_names_clean']).str.lower().str.contains(pattern, na=False)
datadotgov = datadotgov[mask].copy()
datadotgov = datadotgov.drop_duplicates(subset=['legal_name_clean'])

print('Filtered Data.gov rows:', len(datadotgov))
display(datadotgov[datadotgov_cols].head(20))

Filtered Data.gov rows: 12


,Charity_Legal_Name,Other_Organisation_Names,Address_Line_1,Town_City,State,Postcode,Charity_Website
10162,Speld Qld Inc.,NaN,141 Merton Rd,Woolloongabba,QLD,4102,www.speld.org.au
11434,Speld Foundation Of South Australia,NaN,U 2 259 Glen Osmond Rd,Frewville,SA,5063,http://www.speldsa.org.au
22916,Learning Difficulties Australia Inc,NaN,PO Box 76,Mount Waverley,VIC,3149.0,www.ldaustralia.org
32742,Speld-Victoria Inc,NaN,"Unit 2, Level 1, Mt. Alexander Road",Essendon,VIC,3040,https://www.speldvic.org.au
37118,Code Read Dyslexia Network Australia Limited,NaN,Po Box 366,Rockingham,WA,6968,codereadnetwork.org
39485,The Dyslexia Speld Foundation Wa Inc,NaN,10 Broome St,South Perth,WA,6151,www.dsf.net.au
43044,Australian Dyslexia Foundation,NaN,NaN,NaN,NaN,NaN,NaN
44916,Square Pegs Dyslexia Support and Advocacy Inc,NaN,2 Hargrave Pl,Mount Nelson,TAS,7007.0,https://www.squarepegstas.org
46084,Learning Difficulties Coalition Of New South W...,Learning Difficulties Coalition New South Wale...,Unit 21/7 Anella Avenue,Castle Hill,NSW,2154.0,www.ldc.org.au
46999,Speld NSW Inc,NaN,"Suite 2, Level 1",Parramatta,NSW,2150.0,http://www.speldnsw.org.au


## 8. Classify Support Group Records

This block transforms the filtered ACNC rows into the shared map schema. Every row is labeled as a support group, and trusted organizations such as SPELD, DSF, and AUSPELD are highlighted so they can be visually emphasized later on the map.

### How `category` is defined here
All records from the filtered Data.gov / ACNC charity dataset are assigned:
- `category = Support Groups`

This is because these records are not individual medical or therapy services. They are mainly dyslexia associations, learning-difficulty organisations, advocacy groups, or parent/community support organisations.

### How `service_kind` is defined here
All records from this support-group source are assigned:
- `service_kind = Community`

This means they should use the community-style map pin colour, rather than the medical/psychology or education/intervention colours.

### Why `profession_type` is still needed
Even though all of these rows are `Support Groups`, the popup still needs a more specific label. Therefore `profession_type` is used to show labels such as:
- `Dyslexia Support Organisation`
- `Learning Difficulties Association`
- `Community Support Organisation`

### Data.gov / ACNC classification mapping

| Matched text in organisation name | `category` | `service_kind` | `profession_type` |
|---|---|---|---|
| `learning difficulties` | `Support Groups` | `Community` | `Learning Difficulties Association` |
| `dyslexia` | `Support Groups` | `Community` | `Dyslexia Support Organisation` |
| `speld` | `Support Groups` | `Community` | `Dyslexia Support Organisation` |
| `dsf` | `Support Groups` | `Community` | `Dyslexia Support Organisation` |
| `auspeld` | `Support Groups` | `Community` | `Dyslexia Support Organisation` |
| other relevant fallback | `Support Groups` | `Community` | `Community Support Organisation` |

### Highlight rule
`highlighted` is used to mark especially trusted or important dyslexia organisations.

If the organisation name contains one of these terms:
- `speld`
- `dsf`
- `auspeld`

Then:
- `highlighted = Yes`

Otherwise:
- `highlighted = No`

Expected output: a standardized community/support-group dataframe.

In [8]:
# Step 8 code note: reshape ACNC/data.gov rows into the same final shared map_data schema.
classified = datadotgov['legal_name_clean'].apply(classify_datadotgov)
datadotgov[['category', 'service_kind', 'profession_type', 'highlighted']] = pd.DataFrame(classified.tolist(), index=datadotgov.index)

datadotgov_prepared = pd.DataFrame({
    'name': datadotgov['legal_name_clean'],
    'category': datadotgov['category'],
    'service_kind': datadotgov['service_kind'],
    'profession_type': datadotgov['profession_type'],
    'address': datadotgov.apply(
        lambda row: build_address(row.get('Address_Line_1'), title_case_words(row.get('Town_City')), clean(row.get('State')).upper(), clean_postcode(row.get('Postcode'))),
        axis=1,
    ),
    'suburb': datadotgov['Town_City'].apply(title_case_words),
    'state': datadotgov['State'].apply(lambda value: clean(value).upper()),
    'postcode': datadotgov['Postcode'].apply(clean_postcode),
    'latitude': PLACEHOLDER,
    'longitude': PLACEHOLDER,
    'phone': PLACEHOLDER,
    'website': datadotgov['Charity_Website'].apply(normalize_website),
    'source': 'ACNC Registered Charities extract (data.gov.au)',
    'description': 'Community support organisation identified from the ACNC charity register extract.',
    'highlighted': datadotgov['highlighted'],
})

datadotgov_prepared = apply_placeholder(datadotgov_prepared, ['name', 'profession_type', 'address', 'suburb', 'state', 'postcode', 'latitude', 'longitude', 'phone', 'website', 'description'])
display(datadotgov_prepared.head(20))

,name,category,service_kind,profession_type,address,suburb,state,postcode,latitude,longitude,phone,website,source,description,highlighted
10162,Speld Qld Inc.,Support Groups,Community,Dyslexia Support Organisation,"141 Merton Rd, Woolloongabba, QLD, 4102",Woolloongabba,QLD,4102,To be added,To be added,To be added,https://www.speld.org.au,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,Yes
11434,Speld Foundation Of South Australia,Support Groups,Community,Dyslexia Support Organisation,"U 2 259 Glen Osmond Rd, Frewville, SA, 5063",Frewville,SA,5063,To be added,To be added,To be added,http://www.speldsa.org.au,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,Yes
22916,Learning Difficulties Australia Inc,Support Groups,Community,Learning Difficulties Association,"PO Box 76, Mount Waverley, VIC, 3149",Mount Waverley,VIC,3149,To be added,To be added,To be added,https://www.ldaustralia.org,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,No
32742,Speld-Victoria Inc,Support Groups,Community,Dyslexia Support Organisation,"Unit 2, Level 1, Mt. Alexander Road, Essendon,...",Essendon,VIC,3040,To be added,To be added,To be added,https://www.speldvic.org.au,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,Yes
37118,Code Read Dyslexia Network Australia Limited,Support Groups,Community,Dyslexia Support Organisation,"Po Box 366, Rockingham, WA, 6968",Rockingham,WA,6968,To be added,To be added,To be added,https://codereadnetwork.org,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,No
39485,The Dyslexia Speld Foundation Wa Inc,Support Groups,Community,Dyslexia Support Organisation,"10 Broome St, South Perth, WA, 6151",South Perth,WA,6151,To be added,To be added,To be added,https://www.dsf.net.au,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,Yes
43044,Australian Dyslexia Foundation,Support Groups,Community,Dyslexia Support Organisation,To be added,To be added,To be added,To be added,To be added,To be added,To be added,To be added,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,No
44916,Square Pegs Dyslexia Support and Advocacy Inc,Support Groups,Community,Dyslexia Support Organisation,"2 Hargrave Pl, Mount Nelson, TAS, 7007",Mount Nelson,TAS,7007,To be added,To be added,To be added,https://www.squarepegstas.org,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,No
46084,Learning Difficulties Coalition Of New South W...,Support Groups,Community,Learning Difficulties Association,"Unit 21/7 Anella Avenue, Castle Hill, NSW, 2154",Castle Hill,NSW,2154,To be added,To be added,To be added,https://www.ldc.org.au,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,No
46999,Speld NSW Inc,Support Groups,Community,Dyslexia Support Organisation,"Suite 2, Level 1, Parramatta, NSW, 2150",Parramatta,NSW,2150,To be added,To be added,To be added,http://www.speldnsw.org.au,ACNC Registered Charities extract (data.gov.au),Community support organisation identified from...,Yes


## 9. Geocode Missing Coordinates

Map pins need latitude and longitude, so this block attempts to geocode any rows that still lack coordinates. It first tries the full address and then falls back to broader location strings such as suburb, state, and postcode. If no result is found, the row keeps `To be added`.

Expected output: more rows become renderable as pins, while unresolved rows remain clearly marked.

In [9]:
# Step 9 code note: combine both cleaned sources before geocoding and duplicate removal.
combined = pd.concat([healthdirect_prepared, datadotgov_prepared], ignore_index=True)
cache = {}

for idx, row in combined.iterrows():
    if row['latitude'] != PLACEHOLDER and row['longitude'] != PLACEHOLDER:
        continue
    queries = geocode_queries(row['address'], row['suburb'], row['state'], row['postcode'])
    for query in queries:
        latitude, longitude = geocode(query, cache)
        if latitude != PLACEHOLDER and longitude != PLACEHOLDER:
            combined.at[idx, 'latitude'] = latitude
            combined.at[idx, 'longitude'] = longitude
            break

missing_coords = combined[(combined['latitude'] == PLACEHOLDER) | (combined['longitude'] == PLACEHOLDER)]
print('Rows still missing coordinates:', len(missing_coords))
display(missing_coords[['name', 'address', 'suburb', 'state', 'postcode']].head(20))

Rows still missing coordinates: 1


,name,address,suburb,state,postcode
178,Australian Dyslexia Foundation,To be added,To be added,To be added,To be added


## 10. Combine, Deduplicate, and Validate

This block merges the two prepared datasets, removes duplicate records, and runs quality checks. The checks help teammates confirm that the final file follows the agreed data contract before export.

### Final meaning of `category`
`category` is the main filter field for users. It answers: **What kind of support is the parent looking for?**

Allowed values:
- `Assessment`: assessment, identification, or diagnosis-related services.
- `Intervention`: practical support services such as speech pathology, occupational therapy, or literacy-related intervention.
- `Support Groups`: community, advocacy, information, and parent-support organisations.

### Final meaning of `service_kind`
`service_kind` controls the visual group or pin colour on the map. It answers: **What type of provider is this?**

Allowed values:
- `Medical/Psych`: medical, psychology, educational psychology, or paediatrician services.
- `Education`: intervention-style services such as speech pathology or occupational therapy.
- `Community`: dyslexia associations, learning-difficulty organisations, and support groups.

### Why both fields are needed
A single field is not enough because parents need two kinds of information:
- `category` helps them filter by what they need.
- `service_kind` helps them visually understand what type of provider each pin represents.

### Final combined mapping summary

| Source dataset | Matched text | `category` | `service_kind` | `profession_type` |
|---|---|---|---|---|
| Healthdirect | `psychologist`, `psychology` | `Assessment` | `Medical/Psych` | `Psychologist` |
| Healthdirect | `educational psychologist`, `educational psychology` | `Assessment` | `Medical/Psych` | `Educational Psychologist` |
| Healthdirect | `paediatrician`, `pediatrician` | `Assessment` | `Medical/Psych` | `Paediatrician` |
| Healthdirect | `speech pathology`, `speech patholog`, `speech and language` | `Intervention` | `Education` | `Speech Pathologist` |
| Healthdirect | `occupational therapy`, `occupational therapist` | `Intervention` | `Education` | `Occupational Therapist` |
| Data.gov / ACNC | `dyslexia`, `speld`, `dsf`, `auspeld` | `Support Groups` | `Community` | `Dyslexia Support Organisation` |
| Data.gov / ACNC | `learning difficulties` | `Support Groups` | `Community` | `Learning Difficulties Association` |
| Data.gov / ACNC | other relevant fallback | `Support Groups` | `Community` | `Community Support Organisation` |

### Final validation checks
The code checks that the final dataset only contains these values:
- `category`: `Assessment`, `Intervention`, `Support Groups`
- `service_kind`: `Medical/Psych`, `Education`, `Community`

Expected output: one clean dataframe with only approved categories and service kinds.

In [10]:
# Step 10 code note: fill missing coordinates where possible by geocoding address information.
map_data = dedupe_records(combined)
map_data = map_data[FINAL_COLUMNS].copy()

assert set(map_data['category'].unique()) <= {'Assessment', 'Intervention', 'Support Groups'}
assert set(map_data['service_kind'].unique()) <= {'Medical/Psych', 'Education', 'Community'}

print('Final row count:', len(map_data))
print('\nCategory breakdown:')
print(map_data['category'].value_counts())
print('\nService kind breakdown:')
print(map_data['service_kind'].value_counts())
print('\nHighlighted organisations:')
display(map_data[map_data['highlighted'] == 'Yes'][['name', 'category', 'service_kind', 'website']])
print('\nRows with To be added in coordinates:')
display(map_data[(map_data['latitude'] == PLACEHOLDER) | (map_data['longitude'] == PLACEHOLDER)][['name', 'address', 'website']])

Final row count: 160

Category breakdown:
category
Intervention      83
Assessment        65
Support Groups    12
Name: count, dtype: int64

Service kind breakdown:
service_kind
Education        83
Medical/Psych    65
Community        12
Name: count, dtype: int64

Highlighted organisations:


,name,category,service_kind,website
148,Auspeld,Support Groups,Community,https://www.auspeld.org.au
153,Speld Foundation Of South Australia,Support Groups,Community,http://www.speldsa.org.au
154,Speld NSW Inc,Support Groups,Community,http://www.speldnsw.org.au
155,Speld Qld Inc.,Support Groups,Community,https://www.speld.org.au
156,Speld Sa Inc,Support Groups,Community,http://www.speldsa.org.au
157,Speld-Victoria Inc,Support Groups,Community,https://www.speldvic.org.au
159,The Dyslexia Speld Foundation Wa Inc,Support Groups,Community,https://www.dsf.net.au



Rows with To be added in coordinates:


,name,address,website
149,Australian Dyslexia Foundation,To be added,To be added


## 11. Export Final Dataset

This final block writes the cleaned dataframe to `map_data.csv`. That CSV is the handoff artifact for the rest of the project and should be the file used by the second notebook and by teammates working on the frontend and backend later.

Expected output: a new `data/map/map_data.csv` file plus a short summary confirming export success.

In [11]:
# Step 11 code note: deduplicate records and check that final categories/service kinds are valid.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
map_data.to_csv(OUTPUT_PATH, index=False)
print(f'Exported {len(map_data)} rows to {OUTPUT_PATH}')
display(map_data.head(10))

Exported 160 rows to /Users/datong/Documents/5120/Nurodiversity inclusive design/dysLe_TP10/data/map/map_data.csv


,name,category,service_kind,profession_type,address,suburb,state,postcode,latitude,longitude,phone,website,source,description,highlighted
0,24-7Medcare Psychology,Assessment,Medical/Psych,Psychologist,"Ground Floor,737 Burwood Road, Hawthorn, VIC, ...",Hawthorn,VIC,3122,-37.8238804,145.0488157,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
1,AMS Psychology and Consulting Cragieburn,Assessment,Medical/Psych,Psychologist,"Level 1,31 Craigieburn Road, Craigieburn, VIC,...",Craigieburn,VIC,3064,-37.5990452,144.939224,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
2,Adelaide Road Psychology,Assessment,Medical/Psych,Psychologist,"76 Adelaide Road, Gawler South, SA, 5118",Gawler South,SA,5118,-34.61116028,138.74368286,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
3,Balanced Minds Psychology Practice,Assessment,Medical/Psych,Psychologist,"The Italian Forum,SUITE 47A,23 NORTON STREET, ...",Leichhardt,NSW,2040,-33.887211,151.1583777,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
4,Bluebird Psychology,Assessment,Medical/Psych,Psychologist,"Unit 18,15-17 Terminus Street, Castle Hill, NS...",Castle Hill,NSW,2154,-33.73423767,151.00598145,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
5,Central Psychology - Macedon Street Consulting...,Assessment,Medical/Psych,Psychologist,"9 MacEdon Street, Sunbury, VIC, 3429",Sunbury,VIC,3429,-37.57733917,144.73275757,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
6,Child and Adolescent Psychology Services,Assessment,Medical/Psych,Psychologist,"164 THE PARADE, Ascot Vale, VIC, 3032",Ascot Vale,VIC,3032,-37.77180225,144.90969008,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
7,Clear Health Psychology Alkimos,Assessment,Medical/Psych,Psychologist,"Unit 1 & 2,15 Graceful Boulevard, Alkimos, WA,...",Alkimos,WA,6038,-31.6171563,115.6805868,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
8,Cocoon Psychology,Assessment,Medical/Psych,Psychologist,"Level 2,8 KEILOR ROAD, Essendon North, VIC, 3041",Essendon North,VIC,3041,-37.74394247,144.90915579,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
9,Cognition Psychology,Assessment,Medical/Psych,Psychologist,"55 Painswick Street, Berserker, QLD, 4701",Berserker,QLD,4701,-23.36768913,150.52250671,To be added,To be added,Healthdirect NHSD Services Directory 2025,Assessment support identified from Healthdirec...,No
